In [1]:
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Dict, Tuple, Optional

NS = {
    "g": "http://graphml.graphdrawing.org/xmlns",
    "y": "http://www.yworks.com/xml/graphml",
}

def write_graph_overlay_html(
    graphml_path: str,
    background_image_path: str,
    out_html: str = "graph_overlay.html",
    *,
    img_width: Optional[int] = None,   # image width in pixels (set both or neither)
    img_height: Optional[int] = None,  # image height in pixels
    scale: float = 1.0,                # scales graph coordinates (and the image if width/height provided)
    offset_x: float = 0.0,             # shifts graph overlay (use to “align”)
    offset_y: float = 0.0,
    show_labels: bool = False,
    node_size_override: Optional[float] = None,  # if set, force square size in px
    node_fill: str = "#1f77b4",
    node_stroke: str = "#222",
    node_stroke_width: float = 1.0,
    edge_color: str = "#666",
    edge_width: float = 1.0,
) -> None:
    """
    Overlays a yEd GraphML onto a background image in an HTML+SVG viewer.
    Positions come from y:Geometry x,y,width,height. Edges are straight lines between node centers.
    """
    tree = ET.parse(graphml_path)
    root = tree.getroot()

    # --- collect nodes ---
    nodes: Dict[str, Dict] = {}
    minx = miny = float("inf")
    maxx = maxy = float("-inf")

    for n in root.findall(".//g:graph/g:node", NS):
        nid = n.attrib.get("id")
        if not nid:
            continue

        shape = None
        for d in n.findall("./g:data", NS):
            cand = d.find("./y:ShapeNode", NS) or d.find("./y:GenericNode", NS)
            if cand is not None:
                shape = cand
                break
        if shape is None:
            continue

        geom = shape.find("./y:Geometry", NS)
        if geom is None:
            continue

        x = float(geom.attrib.get("x", "0"))
        y = float(geom.attrib.get("y", "0"))
        w = float(geom.attrib.get("width", "0"))
        h = float(geom.attrib.get("height", "0"))

        label_el = shape.find("./y:NodeLabel", NS)
        label_txt = label_el.text.strip() if (label_el is not None and label_el.text) else ""

        nodes[nid] = {"x": x, "y": y, "w": w, "h": h, "label": label_txt}

        minx = min(minx, x); miny = min(miny, y)
        maxx = max(maxx, x + w); maxy = max(maxy, y + h)

    # --- collect edges ---
    edges: Dict[str, Tuple[str, str]] = {}
    for e in root.findall(".//g:graph/g:edge", NS):
        sid = e.attrib.get("source")
        tid = e.attrib.get("target")
        if sid in nodes and tid in nodes:
            edges[e.attrib.get("id", f"{sid}->{tid}")] = (sid, tid)

    # --- decide canvas size ---
    # If img_width/height provided, we’ll size the <image> accordingly.
    # Otherwise, we’ll size the SVG to fit the graph bbox (after scale) and still draw the image at its natural size (browser will display it).
    graph_w = (maxx - minx) * scale
    graph_h = (maxy - miny) * scale

    # If no explicit image size, use graph bbox as the SVG size to start; you can still pan/scroll.
    svg_w = int(img_width if img_width else max(1, graph_w))
    svg_h = int(img_height if img_height else max(1, graph_h))

    # --- build SVG ---
    def esc(s: str) -> str:
        return (s or "").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

    # Background image element; if width/height are given, we set them and they’ll scale with CSS pixels.
    img_attr = []
    img_attr.append(f'href="{esc(Path(background_image_path).as_posix())}"')
    if img_width:
        img_attr.append(f'width="{img_width}"')
    if img_height:
        img_attr.append(f'height="{img_height}"')

    # A group to transform the graph overlay (offset + scale)
    # Note: we translate by (-minx, -miny) so graph's top-left starts at 0,0, then apply offset & scale.
    overlay_transform = f"translate({offset_x:.2f},{offset_y:.2f}) scale({scale:.8f}) translate({-minx:.2f},{-miny:.2f})"

    # Build edge lines first (behind nodes)
    edge_lines = []
    for _, (sid, tid) in edges.items():
        s = nodes[sid]; t = nodes[tid]
        sx = s["x"] + s["w"] / 2.0
        sy = s["y"] + s["h"] / 2.0
        tx = t["x"] + t["w"] / 2.0
        ty = t["y"] + t["h"] / 2.0
        edge_lines.append(
            f'<line x1="{sx:.2f}" y1="{sy:.2f}" x2="{tx:.2f}" y2="{ty:.2f}" '
            f'stroke="{edge_color}" stroke-width="{edge_width:.2f}" stroke-linecap="round" />'
        )

    # Build node squares
    node_rects = []
    for nid, ninfo in nodes.items():
        w = node_size_override if node_size_override is not None else ninfo["w"]
        h = node_size_override if node_size_override is not None else ninfo["h"]
        # Center square at node center:
        cx = ninfo["x"] + ninfo["w"]/2.0
        cy = ninfo["y"] + ninfo["h"]/2.0
        x = cx - (w/2.0)
        y = cy - (h/2.0)
        rect = (
            f'<rect x="{x:.2f}" y="{y:.2f}" width="{w:.2f}" height="{h:.2f}" '
            f'fill="{node_fill}" stroke="{node_stroke}" stroke-width="{node_stroke_width:.2f}" rx="0" ry="0" />'
        )
        if show_labels and ninfo["label"]:
            # render label slightly below the square
            label = esc(ninfo["label"])
            label_y = y + h + 10
            rect += f'\n  <text x="{cx:.2f}" y="{label_y:.2f}" text-anchor="middle" font-size="12">{label}</text>'
        node_rects.append(rect)

    html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8" />
<title>Graph Overlay</title>
<style>
  body {{ margin: 0; background:#111; color:#ddd; font-family: system-ui, sans-serif; }}
  .viewport {{
    width: 100vw;
    height: 100vh;
    overflow: scroll;
    background: #111;
  }}
  svg {{ background: #111; }}
</style>
</head>
<body>
<div class="viewport">
  <svg xmlns="http://www.w3.org/2000/svg"
       xmlns:xlink="http://www.w3.org/1999/xlink"
       width="{svg_w}" height="{svg_h}" viewBox="0 0 {svg_w} {svg_h}">
    <!-- Background image -->
    <image {' '.join(img_attr)} x="0" y="0" />
    <!-- Graph overlay -->
    <g transform="{overlay_transform}">
      <!-- edges -->
      {''.join(edge_lines)}
      <!-- nodes -->
      {''.join(node_rects)}
    </g>
  </svg>
</div>
</body>
</html>
"""
    Path(out_html).write_text(html, encoding="utf-8")
    print(f"Wrote overlay to: {out_html}")
    print("Open it in a browser; use mouse wheel / trackpad to zoom (Ctrl+wheel) and scroll to pan.")


In [3]:
write_graph_overlay_html(
    graphml_path="../mapGraphs/graphml/HIMCM_graph_cleaned.graphml",
    background_image_path="../map_grid.png",
    out_html="../mapGraphs/html/overlay.html",
    img_width=1954, img_height=1566,   # set to your image’s pixel size
    scale=1.0,
    offset_x=-40, offset_y=-40,        # undo the margins from the snap step
    show_labels=False,
    node_size_override=12.0
)


Wrote overlay to: ../mapGraphs/html/overlay.html
Open it in a browser; use mouse wheel / trackpad to zoom (Ctrl+wheel) and scroll to pan.


/tmp/ipykernel_340707/445210168.py:47: DeprecationWarning: Testing an element's truth value will raise an exception in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  cand = d.find("./y:ShapeNode", NS) or d.find("./y:GenericNode", NS)
